# Plenum Prometheus | First free cloud inference trial
Open in **Kaggle Notebooks** on your iPad. Choose the free **GPU T4 x2** accelerator and enable Internet if available. This notebook is **not** a persistent service and must not contain confidential Plenum intellectual property, user videos, access tokens or private source documents.
It checks real GPU availability before any download. The proposed community GGUF quantization is about **19.1 GB**, requiring additional GPU/CPU memory. Cloud quotas, temporary storage, CUDA compiler availability, or model compatibility can prevent a successful run. **No AGI is implied.**
The download and engine build are behind the explicit `APPROVE_FREE_GPU_RUN` flag. The fixed smoke test is intentionally trivial and cannot demonstrate broad generalization. Do not post raw private inputs to public notebooks.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time, hashlib
from datetime import datetime, timezone

print("Plenum Prometheus: free-cloud hardware preflight only")
print("Python:", sys.version.split()[0])
print("GPU command present:", bool(shutil.which("nvidia-smi")))
if shutil.which("nvidia-smi"):
    gpus = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True, timeout=25, check=True
    ).stdout.strip()
    print("Available GPUs (MiB):\n" + gpus)
else:
    gpus = ""
for candidate in ["/kaggle/temp", "/kaggle/working"]:
    if Path(candidate).is_dir():
        print(candidate, "free GiB:", round(shutil.disk_usage(candidate).free / (1024**3), 2))
print("CUDA compiler available:", bool(shutil.which("nvcc")))
print("No model or engine has been downloaded or executed.")

In [ ]:
# HUMAN-APPROVED MANUAL EXPERIMENT ONLY. Read the notes and quotas first.
APPROVE_FREE_GPU_RUN = False
MODEL_REPO = "ggml-org/Qwen3.6-27B-GGUF"
MODEL_FILENAME = "Qwen3.6-27B-Q4_K_M.gguf"
LLAMA_CPP_TAG = "b12143"
PUBLIC_PROMPT = "Compute 17 + 25. Reply with only the integer."
if not APPROVE_FREE_GPU_RUN:
    print("STOP: change APPROVE_FREE_GPU_RUN=True after checking GPU, disk, quota and Internet. No downloads or fees were triggered.")

In [ ]:
if APPROVE_FREE_GPU_RUN:
    import re
    if not (shutil.which("nvcc") and shutil.which("cmake") and shutil.which("git")):
        raise RuntimeError("Kaggle CUDA compiler, cmake and git are required; stop, do not silently downgrade.")
    gpu_lines = [line.strip() for line in gpus.splitlines() if line.strip()]
    if len(gpu_lines) != 2 or any("T4" not in line for line in gpu_lines):
        raise RuntimeError("This preregistered trial expects exactly two Tesla T4 GPUs.")
    temp = Path("/kaggle/temp")
    if not temp.is_dir() or shutil.disk_usage(temp).free < 25 * 1024**3:
        raise RuntimeError("Need >=25 GiB free TEMP space before downloading a 19.1 GB model.")
    import importlib.util
    if importlib.util.find_spec("huggingface_hub") is None:
        subprocess.run([sys.executable, "-m", "pip", "install",
                        "huggingface_hub>=0.29,<2"], check=True, timeout=240)
    from huggingface_hub import HfApi, hf_hub_download
    model_info = HfApi().model_info(MODEL_REPO)
    model_revision = model_info.sha
    if not re.fullmatch(r"[0-9a-f]{40}", model_revision):
        raise RuntimeError("Cannot pin an exact Hugging Face source revision.")
    print("Resolved exact source revision:", model_revision)
    work = temp / "prometheus_qwen36_27b"
    work.mkdir(exist_ok=True, parents=True)
    source_dir = work / "llama.cpp"
    if source_dir.exists():
        raise RuntimeError("Existing source directory: use a fresh notebook instance to avoid ambiguity.")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", LLAMA_CPP_TAG,
                    "https://github.com/ggml-org/llama.cpp.git", str(source_dir)],
                   check=True, timeout=300)
    engine_commit = subprocess.run(
        ["git", "-C", str(source_dir), "rev-parse", "HEAD"],
        capture_output=True, check=True, text=True).stdout.strip()
    subprocess.run(["cmake", "-S", str(source_dir), "-B", str(source_dir / "build"),
                    "-DGGML_CUDA=ON", "-DCMAKE_BUILD_TYPE=Release"],
                   check=True, timeout=240)
    subprocess.run(["cmake", "--build", str(source_dir / "build"),
                    "--target", "llama-cli", "-j", "2"], check=True, timeout=1800)
    engine = source_dir / "build" / "bin" / "llama-cli"
    if not engine.is_file():
        raise RuntimeError("Expected compiled llama-cli was not created.")
    print("Pinned llama.cpp revision:", engine_commit)
    model_path = Path(hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILENAME,
                    revision=model_revision, local_dir=str(work / "model")))
    if model_path.stat().st_size < 15 * 10**9 or model_path.open("rb").read(4) != b"GGUF":
        raise RuntimeError("Downloaded model has unexpected size or header.")
    def filehash(p):
        h = hashlib.sha256()
        with Path(p).open("rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                h.update(chunk)
        return h.hexdigest()
    model_hash = filehash(model_path)
    print("Model file SHA256:", model_hash)
    cli = [str(engine), "-m", str(model_path),
           "--split-mode", "layer", "--tensor-split", "1,1",
           "-ngl", "99", "-c", "2048", "-n", "64", "-t", "4",
           "--temp", "0", "--seed", "1", "--single-turn",
           "--no-conversation", "--no-display-prompt",
           "-p", PUBLIC_PROMPT]
    start = time.monotonic()
    try:
        run = subprocess.run(cli, capture_output=True, text=True, timeout=600,
                             env={k: os.environ[k] for k in ("PATH", "LANG", "LC_ALL",
                                                            "CUDA_VISIBLE_DEVICES")
                                  if k in os.environ})
        elapsed = time.monotonic() - start
    except subprocess.TimeoutExpired:
        raise RuntimeError("Fixed inference timeout exceeded; preserve failure.")
    response = run.stdout if run.stdout else ""
    # This is deliberately a strict public smoke case, NOT a general-intelligence benchmark.
    strict_match = (run.returncode == 0 and response.strip() == "42")
    report = {
        "schema": "plenum-kaggle-free-pilot-v1",
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "model_repo": MODEL_REPO, "model_revision": model_revision,
        "model_gguf": MODEL_FILENAME,
        "model_file_sha256": model_hash,
        "engine_release": LLAMA_CPP_TAG, "engine_commit": engine_commit,
        "gpu_description": gpu_lines,
        "inference_exit_code": run.returncode,
        "elapsed_seconds": round(elapsed, 3),
        "stdout_sha256": hashlib.sha256(response.encode("utf-8")).hexdigest(),
        "stdout_characters": len(response),
        "public_smoke_exact_match": strict_match,
        "independent_scientific_benchmark_passed": False,
        "AGI_demonstrated": False, "paid_api_calls": 0,
        "private_research_input_used": False,
        "always_on_service_deployed": False
    }
    out = Path("/kaggle/working/prometheus_free_cloud_report.json")
    out.write_text(json.dumps(report, indent=2) + "\n")
    print("Report saved:", out)
    print("Process exit:", run.returncode, "| strict public smoke exact:", strict_match)
    print("If false, inspect output locally without claiming model failure on unseen benchmarks.")
    print("STOP: no background services, no self-training, no publication.")
else:
    print("No model inference requested.")

When finished, **stop the GPU session** to conserve free quota. Do not confuse this smoke test with the existing independently scored Inspect benchmark suite. Preserve the real result, including failures, before choosing a stronger model or permanent hosting.
Provider docs: https://www.kaggle.com/docs/notebooks ; model: https://huggingface.co/ggml-org/Qwen3.6-27B-GGUF ; engine: https://github.com/ggml-org/llama.cpp/releases/tag/b12143
